# Notebook 2: Mathematical Optimization — PuLP Mixed-Integer Linear Programming (MILP)

**Project:** Integrated Hospital Operations Intelligence, Resource Optimization & Patient Continuity Platform  
**Academic Course:** MCA III-Semester — Machine Learning Capstone  
**Focus:** Module 5 — Predictive Resource Planning & Optimization Engine  
**Objective:** Formulate, solve, and analyze a Mixed-Integer Linear Programming (MILP) model in PuLP for dynamic bed and acuity allocation under non-negotiable clinical safety and nurse-to-patient staffing constraints.

## 1. Problem Formulation & Mathematical Model

### Decision Variables
Let $x_{i, j} \in \{0, 1\}$ denote the binary assignment variable:
$$x_{i, j} = 1 \quad \text{if patient } i \text{ is assigned to bed } j, \quad 0 \text{ otherwise.}$$

### Objective Function
Minimize total patient waiting delay, inter-ward transfer penalties, and unmatched patient penalties:
$$\min \quad \sum_{i} \sum_{j} \left( \text{WaitHours}_i \cdot \text{PriorityWeight}_i + \text{WardMismatchCost}_{i, j} \right) x_{i, j} + \sum_{i} M \cdot (1 - \sum_{j} x_{i, j})$$
where $M \gg 0$ is a large penalty for leaving a patient unallocated.

### Non-Negotiable Hard Constraints
1. **Single Bed Assignment:** $\sum_{j} x_{i, j} \le 1 \quad \forall i$
2. **Single Patient Per Bed:** $\sum_{i} x_{i, j} \le 1 \quad \forall j$
3. **Physical Bed State:** $x_{i, j} = 0 \quad \text{if } \text{BedState}_j \neq \text{'Ready'}$
4. **Acuity & Specialty Protocol:** High acuity patients (ESI 1 & 2) must be assigned strictly to ICU/Acute beds:
   $$x_{i, j} = 0 \quad \text{if } \text{Acuity}_i = 1 \text{ and } \text{Specialty}_j \neq \text{'ICU'}$$
5. **Isolation Protocol:** Patients with airborne/contact infection holds must be placed in isolation-certified beds:
   $$x_{i, j} = 0 \quad \text{if } \text{NeedsIsolation}_i = 1 \text{ and } \text{IsIsolationCapable}_j = 0$$
6. **Operational Staffed Ceiling:** Total occupied beds in ward $w$ cannot exceed active nurse capacity:
   $$\sum_{j \in \text{Ward}_w} \sum_{i} x_{i, j} + \text{ExistingCensus}_w \le \text{NursesOnDuty}_w \times \text{MaxRatio}_w$$

## 2. Environment Setup & PuLP Solver Initialization

In [ ]:
import pulp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print(f"[*] PuLP Version: {pulp.__version__}")
print(f"[*] Available Solvers: {pulp.listSolvers(onlyAvailable=True)}")

## 3. Defining Ward Infrastructure & Patient Influx

In [ ]:
# Sample Wards
wards = {
    "ICU_NORTH": {"specialty": "ICU", "total_beds": 6, "ready_beds": 4, "current_census": 2, "nurses": 4, "ratio": 1.0, "isolation_beds": 2},
    "MED_WARD_A": {"specialty": "General", "total_beds": 15, "ready_beds": 8, "current_census": 6, "nurses": 2, "ratio": 4.0, "isolation_beds": 1},
    "SURG_WARD_A": {"specialty": "Surgical", "total_beds": 12, "ready_beds": 6, "current_census": 5, "nurses": 2, "ratio": 4.0, "isolation_beds": 0},
}

# Sample Queue of Patients Awaiting Bed Placement
patients = [
    {"id": "P01", "esi": 1, "acuity": "Critical", "specialty": "ICU", "isolation": True, "wait_hours": 1.5},
    {"id": "P02", "esi": 2, "acuity": "Severe", "specialty": "ICU", "isolation": False, "wait_hours": 2.0},
    {"id": "P03", "esi": 3, "acuity": "Moderate", "specialty": "General", "isolation": True, "wait_hours": 3.5},
    {"id": "P04", "esi": 3, "acuity": "Moderate", "specialty": "General", "isolation": False, "wait_hours": 1.2},
    {"id": "P05", "esi": 4, "acuity": "Low", "specialty": "General", "isolation": False, "wait_hours": 4.0},
    {"id": "P06", "esi": 3, "acuity": "Post-Op", "specialty": "Surgical", "isolation": False, "wait_hours": 1.8},
    {"id": "P07", "esi": 2, "acuity": "Severe", "specialty": "ICU", "isolation": False, "wait_hours": 0.8},
]

print(f"[*] Total Patients in Queue: {len(patients)}")
pd.DataFrame(patients)

## 4. Constructing and Solving the MILP Model in PuLP

In [ ]:
def solve_bed_allocation(patients_list, wards_dict):
    model = pulp.LpProblem("Hospital_Bed_Allocation", pulp.LpMinimize)
    
    # Indices
    P = [p["id"] for p in patients_list]
    W = list(wards_dict.keys())
    
    # Decision variables: x[p, w] = 1 if patient p assigned to ward w
    x = pulp.LpVariable.dicts("alloc", ((p, w) for p in P for w in W), cat=pulp.LpBinary)
    
    # Slack variables: unassigned[p] = 1 if patient p cannot be safely assigned
    unassigned = pulp.LpVariable.dicts("unassigned", P, cat=pulp.LpBinary)
    
    # Big-M penalty for leaving patients unassigned
    BIG_M = 1000.0
    
    # Objective: Minimize wait times + mismatch penalties + Big-M unallocated
    obj = pulp.lpSum(
        x[p["id"], w] * (p["wait_hours"] * (6 - p["esi"]) + (0 if p["specialty"] == wards_dict[w]["specialty"] else 50))
        for p in patients_list for w in W
    ) + pulp.lpSum(unassigned[p["id"]] * BIG_M * (6 - p["esi"]) for p in patients_list)
    
    model += obj
    
    # Constraints
    # 1. Assignment or Unassigned
    for p in P:
        model += pulp.lpSum(x[p, w] for w in W) + unassigned[p] == 1, f"AssignOrUnassigned_{p}"
        
    # 2. Hard Acuity matching: ESI 1 and 2 must only go to ICU
    for p in patients_list:
        if p["esi"] in [1, 2]:
            for w in W:
                if wards_dict[w]["specialty"] != "ICU":
                    model += x[p["id"], w] == 0, f"HardAcuity_{p['id']}_{w}"
                    
    # 3. Hard Isolation constraint
    for p in patients_list:
        if p["isolation"]:
            for w in W:
                if wards_dict[w]["isolation_beds"] == 0:
                    model += x[p["id"], w] == 0, f"HardIsolation_{p['id']}_{w}"
                    
    # 4. Ward Physical Bed & Staffed Ceiling Constraint
    for w in W:
        ward_info = wards_dict[w]
        staffed_capacity = ward_info["nurses"] * ward_info["ratio"]
        usable_capacity = min(ward_info["ready_beds"], max(0, staffed_capacity - ward_info["current_census"]))
        model += pulp.lpSum(x[p, w] for p in P) <= usable_capacity, f"WardUsableCapacity_{w}"
        
    # Solve using CBC or default solver
    status = model.solve(pulp.PULP_CBC_CMD(msg=0))
    
    allocations = []
    for p in patients_list:
        assigned_ward = None
        for w in W:
            if pulp.value(x[p["id"], w]) == 1:
                assigned_ward = w
                break
        allocations.append({
            "Patient": p["id"],
            "ESI": p["esi"],
            "Specialty": p["specialty"],
            "Isolation": p["isolation"],
            "AssignedWard": assigned_ward if assigned_ward else "UNASSIGNED (Infeasible)"
        })
        
    return pulp.LpStatus[status], pulp.value(model.objective), pd.DataFrame(allocations)

status, obj_val, alloc_df = solve_bed_allocation(patients, wards)
print(f"[+] Solver Status: {status}")
print(f"[+] Optimal Objective Value: {obj_val:.2f}")
alloc_df

## 5. Deliberate Infeasibility Experiment (Safety Guardrail Verification)

To verify our ethical rule that **hard constraints are never relaxed**, we simulate a critical surge of 8 ICU patients when only 2 staffed ICU beds exist.

In [ ]:
# Overwhelming ICU surge: 8 Critical ESI 1 patients
surge_patients = [
    {"id": f"SURGE_{i:02d}", "esi": 1, "acuity": "Critical", "specialty": "ICU", "isolation": False, "wait_hours": 2.0}
    for i in range(1, 9)
]

# Tight ICU capacity: only 2 usable beds
tight_wards = {
    "ICU_NORTH": {"specialty": "ICU", "total_beds": 6, "ready_beds": 2, "current_census": 4, "nurses": 2, "ratio": 1.0, "isolation_beds": 1},
    "MED_WARD_A": {"specialty": "General", "total_beds": 15, "ready_beds": 10, "current_census": 5, "nurses": 4, "ratio": 4.0, "isolation_beds": 2},
}

status_surge, obj_surge, surge_alloc = solve_bed_allocation(surge_patients, tight_wards)
print(f"[*] Solver Result during Extreme Surge: {status_surge}")

unassigned_count = (surge_alloc["AssignedWard"] == "UNASSIGNED (Infeasible)").sum()
print(f"[!] Correctly Flagged Unassigned Patients: {unassigned_count} / {len(surge_patients)}")
surge_alloc

## 6. What-If Scenario Analysis: Staff Shortage Stress Curve

In [ ]:
# Sweep nurse absenteeism from 0% to 50%
absentee_rates = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
unassigned_trajectory = []

for rate in absentee_rates:
    test_wards = {
        k: {
            **v,
            "nurses": max(1, int(np.round(v["nurses"] * (1.0 - rate))))
        }
        for k, v in wards.items()
    }
    _, _, res = solve_bed_allocation(patients, test_wards)
    unassigned_count = (res["AssignedWard"] == "UNASSIGNED (Infeasible)").sum()
    unassigned_trajectory.append(unassigned_count)

plt.figure(figsize=(8, 4.5))
plt.plot([r * 100 for r in absentee_rates], unassigned_trajectory, marker="s", color="#ef4444", linewidth=2.5)
plt.title("Impact of Nurse Absenteeism on Unplaced Patient Queue", fontsize=12, fontweight="bold")
plt.xlabel("Nurse Absenteeism Rate (%)")
plt.ylabel("Unassigned Patients (Deficit)")
plt.ylim(-0.5, max(unassigned_trajectory) + 2)
plt.tight_layout()
plt.show()

## 7. Viva Defense Evaluation Summary

1. **Mathematical Rigor:** The PuLP MILP formulation simultaneously evaluates patient wait times, clinical specialty requirements, and nurse staffing ratios.
2. **Zero False Feasibility:** The solver never places an ICU patient in a general medical bed or ignores nurse staffing ceilings just to force a feasible solution.
3. **Decision-Support Guardrail:** When capacity is exceeded, the system outputs an actionable deficit audit rather than autonomous forced reassignments.